In [45]:
import pandas as pd
import numpy as np
import re
import joblib
import spacy

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    accuracy_score,
    confusion_matrix
)

print("Libraries loaded successfully")

Libraries loaded successfully


In [46]:
df = pd.read_csv("clean_jobs.csv")

print("Dataset loaded successfully")
print("Total jobs:", len(df))

Dataset loaded successfully
Total jobs: 10100


In [47]:
test_df = df.sample(
    100,
    random_state=42
).copy()

test_df = test_df[
    ["job_title", "clean_description"]
].reset_index(drop=True)

print("Evaluation jobs:", len(test_df))

Evaluation jobs: 100


In [48]:
skill_list = [
    "python",
    "java",
    "c++",
    "javascript",
    "typescript",
    "sql",
    "mysql",
    "postgresql",
    "mongodb",
    "power bi",
    "tableau",
    "excel",
    "pandas",
    "numpy",
    "scikit-learn",
    "tensorflow",
    "pytorch",
    "machine learning",
    "deep learning",
    "natural language processing",
    "aws",
    "azure",
    "gcp",
    "spark",
    "hadoop",
    "docker",
    "kubernetes",
    "git"
]

print("Number of skills:", len(skill_list))

Number of skills: 28


In [49]:
def get_actual_skills(text):
    text = text.lower()
    found = []

    for skill in skill_list:
        if skill in text:
            found.append(skill)

    return sorted(set(found))


test_df["actual_skills"] = test_df[
    "clean_description"
].apply(get_actual_skills)

print("Reference skills created")

Reference skills created


In [50]:
test_df[
    ["job_title", "actual_skills"]
].head(10)

,job_title,actual_skills
0,Backend Developer,"[docker, postgresql, sql]"
1,Data Scientist,"[aws, machine learning, numpy, pandas, sql]"
2,Data Analyst,"[excel, pandas, power bi, python, sql]"
3,QA Engineer,"[git, sql]"
4,Business Analyst,"[excel, power bi, sql, tableau]"
5,Data Analyst,"[excel, pandas, python, sql, tableau]"
6,Data Scientist,"[aws, machine learning, python, sql]"
7,Machine Learning Engineer,"[docker, machine learning, python, pytorch, sq..."
8,Business Analyst,"[excel, power bi, sql, tableau]"
9,Frontend Developer,"[git, java, javascript, typescript]"


In [51]:
def calculate_metrics(actual, predicted, all_skills):

    y_true = []
    y_pred = []

    for actual_skills, predicted_skills in zip(
        actual, predicted
    ):

        actual_set = set(actual_skills)
        predicted_set = set(predicted_skills)

        for skill in all_skills:

            y_true.append(
                1 if skill in actual_set else 0
            )

            y_pred.append(
                1 if skill in predicted_set else 0
            )

    precision = precision_score(
        y_true,
        y_pred,
        zero_division=0
    )

    recall = recall_score(
        y_true,
        y_pred,
        zero_division=0
    )

    f1 = f1_score(
        y_true,
        y_pred,
        zero_division=0
    )

    accuracy = accuracy_score(
        y_true,
        y_pred
    )

    cm = confusion_matrix(
        y_true,
        y_pred
    )

    return precision, recall, f1, accuracy, cm

In [52]:
def dictionary_extract(text):

    text = text.lower()
    found = []

    for skill in skill_list:
        if skill in text:
            found.append(skill)

    return sorted(set(found))


test_df["dictionary_skills"] = test_df[
    "clean_description"
].apply(dictionary_extract)

In [53]:
dictionary_metrics = calculate_metrics(
    test_df["actual_skills"],
    test_df["dictionary_skills"],
    skill_list
)

print("DICTIONARY")
print("Precision:", round(dictionary_metrics[0], 3))
print("Recall:", round(dictionary_metrics[1], 3))
print("F1 Score:", round(dictionary_metrics[2], 3))
print("Accuracy:", round(dictionary_metrics[3], 3))

print("\nConfusion Matrix:")
print(dictionary_metrics[4])

DICTIONARY
Precision: 1.0
Recall: 1.0
F1 Score: 1.0
Accuracy: 1.0

Confusion Matrix:
[[2412    0]
 [   0  388]]


In [55]:
patterns = {
    "python": r"\bpython(?:\s*3)?(?:\s+programming)?\b",
    "java": r"\bjava\b",
    "c++": r"\bc\+\+\b",
    "javascript": r"\bjavascript\b",
    "typescript": r"\btypescript\b",
    "sql": r"\bsql\b",
    "mysql": r"\bmysql\b",
    "postgresql": r"\b(?:postgres|postgresql)\b",
    "mongodb": r"\b(?:mongodb|mongo)\b",
    "power bi": r"\bpower\s*bi\b",
    "tableau": r"\btableau\b",
    "excel": r"\b(?:excel|ms\s+excel)\b",
    "pandas": r"\bpandas\b",
    "numpy": r"\bnumpy\b",
    "scikit-learn": r"\b(?:scikit[- ]learn|sklearn)\b",
    "tensorflow": r"\btensorflow\b",
    "pytorch": r"\bpytorch\b",
    "machine learning": r"\bmachine\s+learning\b",
    "deep learning": r"\bdeep\s+learning\b",
    "natural language processing": r"\bnatural\s+language\s+processing\b",
    "aws": r"\baws\b",
    "azure": r"\bazure\b",
    "gcp": r"\b(?:gcp|google\s+cloud)\b",
    "spark": r"\b(?:spark|apache\s+spark)\b",
    "hadoop": r"\b(?:hadoop|apache\s+hadoop)\b",
    "docker": r"\bdocker\b",
    "kubernetes": r"\b(?:kubernetes|k8s)\b",
    "git": r"\bgit\b"
}

In [57]:
regex_metrics = calculate_metrics(
    test_df["actual_skills"],
    test_df["regex_skills"],
    skill_list
)

print("REGEX")
print("Precision:", round(regex_metrics[0], 3))
print("Recall:", round(regex_metrics[1], 3))
print("F1 Score:", round(regex_metrics[2], 3))
print("Accuracy:", round(regex_metrics[3], 3))

print("\nConfusion Matrix:")
print(regex_metrics[4])

REGEX
Precision: 0.962
Recall: 0.977
F1 Score: 0.969
Accuracy: 0.991

Confusion Matrix:
[[2397   15]
 [   9  379]]


In [58]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    max_features=1000,
    stop_words="english",
    ngram_range=(1, 3)
)

tfidf_matrix = tfidf.fit_transform(
    test_df["clean_description"]
)

terms = tfidf.get_feature_names_out()

print("Number of TF-IDF terms:", len(terms))

Number of TF-IDF terms: 971


In [59]:
def tfidf_extract(text, top_n=20):

    vector = tfidf.transform([text])

    scores = vector.toarray()[0]

    ranked_indices = scores.argsort()[::-1]

    top_terms = []

    for i in ranked_indices[:top_n]:

        if scores[i] > 0:
            top_terms.append(
                terms[i].lower()
            )

    found = []

    for skill in skill_list:

        if skill in top_terms:
            found.append(skill)

    return sorted(set(found))

In [60]:
test_df["tfidf_skills"] = test_df[
    "clean_description"
].apply(tfidf_extract)

In [61]:
tfidf_metrics = calculate_metrics(
    test_df["actual_skills"],
    test_df["tfidf_skills"],
    skill_list
)

print("TF-IDF")
print("Precision:", round(tfidf_metrics[0], 3))
print("Recall:", round(tfidf_metrics[1], 3))
print("F1 Score:", round(tfidf_metrics[2], 3))
print("Accuracy:", round(tfidf_metrics[3], 3))

print("\nConfusion Matrix:")
print(tfidf_metrics[4])

TF-IDF
Precision: 1.0
Recall: 0.245
F1 Score: 0.393
Accuracy: 0.895

Confusion Matrix:
[[2412    0]
 [ 293   95]]


In [62]:
nlp = spacy.load("en_core_web_sm")

print("spaCy NER model loaded")

spaCy NER model loaded


In [63]:
def ner_extract(text):

    doc = nlp(text)

    found = []

    for ent in doc.ents:

        entity = ent.text.lower().strip()

        if entity in skill_list:
            found.append(entity)

    return sorted(set(found))


test_df["ner_skills"] = test_df[
    "clean_description"
].apply(ner_extract)

In [64]:
ner_metrics = calculate_metrics(
    test_df["actual_skills"],
    test_df["ner_skills"],
    skill_list
)

print("NER")
print("Precision:", round(ner_metrics[0], 3))
print("Recall:", round(ner_metrics[1], 3))
print("F1 Score:", round(ner_metrics[2], 3))
print("Accuracy:", round(ner_metrics[3], 3))

print("\nConfusion Matrix:")
print(ner_metrics[4])

NER
Precision: 1.0
Recall: 0.021
F1 Score: 0.04
Accuracy: 0.864

Confusion Matrix:
[[2412    0]
 [ 380    8]]


In [65]:
ml_model = joblib.load(
    "skill_classifier.pkl"
)

print("ML model loaded successfully")

ML model loaded successfully


In [66]:
def ml_extract(text):

    found = []

    text_lower = text.lower()

    for skill in skill_list:

        if skill in text_lower:

            prediction = ml_model.predict(
                [skill]
            )[0]

            # The classifier recognizes this
            # candidate as a known project category.
            if prediction in [
                "SKILL",
                "CLOUD",
                "DATABASE",
                "BI_TOOL",
                "TECHNOLOGY"
            ]:
                found.append(skill)

    return sorted(set(found))


test_df["ml_skills"] = test_df[
    "clean_description"
].apply(ml_extract)

In [67]:
ml_metrics = calculate_metrics(
    test_df["actual_skills"],
    test_df["ml_skills"],
    skill_list
)

print("ML MODEL")
print("Precision:", round(ml_metrics[0], 3))
print("Recall:", round(ml_metrics[1], 3))
print("F1 Score:", round(ml_metrics[2], 3))
print("Accuracy:", round(ml_metrics[3], 3))

print("\nConfusion Matrix:")
print(ml_metrics[4])

ML MODEL
Precision: 1.0
Recall: 1.0
F1 Score: 1.0
Accuracy: 1.0

Confusion Matrix:
[[2412    0]
 [   0  388]]


In [68]:
from transformers import pipeline

transformer_ner = pipeline(
    "ner",
    model="./skill_bert_model_final",
    tokenizer="./skill_bert_model_final",
    aggregation_strategy="simple"
)

print("Transformer model loaded successfully")

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

Transformer model loaded successfully


In [69]:
def transformer_extract(text):

    results = transformer_ner(text)

    found = []

    for entity in results:

        word = entity["word"].lower().strip()

        # Remove WordPiece markers
        word = word.replace("##", "")

        if word in skill_list:
            found.append(word)

    return sorted(set(found))

In [70]:
def transformer_extract(text):

    results = transformer_ner(text)

    found = []

    for entity in results:

        word = entity["word"].lower().strip()
        word = word.replace("##", "")

        if not any(c.isalnum() for c in word):
            continue

        for skill in skill_list:

            skill_clean = skill.replace(" ", "")

            if word == skill_clean:
                found.append(skill)

    return sorted(set(found))

In [71]:
test_df["transformer_skills"] = test_df[
    "clean_description"
].apply(transformer_extract)

In [72]:
transformer_metrics = calculate_metrics(
    test_df["actual_skills"],
    test_df["transformer_skills"],
    skill_list
)

print("TRANSFORMER")
print("Precision:", round(transformer_metrics[0], 3))
print("Recall:", round(transformer_metrics[1], 3))
print("F1 Score:", round(transformer_metrics[2], 3))
print("Accuracy:", round(transformer_metrics[3], 3))

print("\nConfusion Matrix:")
print(transformer_metrics[4])

TRANSFORMER
Precision: 1.0
Recall: 0.84
F1 Score: 0.913
Accuracy: 0.978

Confusion Matrix:
[[2412    0]
 [  62  326]]


In [73]:
model_comparison = pd.DataFrame([
    
    {
        "Method": "Dictionary",
        "Precision": dictionary_metrics[0],
        "Recall": dictionary_metrics[1],
        "F1 Score": dictionary_metrics[2],
        "Accuracy": dictionary_metrics[3]
    },

    {
        "Method": "Regex",
        "Precision": regex_metrics[0],
        "Recall": regex_metrics[1],
        "F1 Score": regex_metrics[2],
        "Accuracy": regex_metrics[3]
    },

    {
        "Method": "TF-IDF",
        "Precision": tfidf_metrics[0],
        "Recall": tfidf_metrics[1],
        "F1 Score": tfidf_metrics[2],
        "Accuracy": tfidf_metrics[3]
    },

    {
        "Method": "NER",
        "Precision": ner_metrics[0],
        "Recall": ner_metrics[1],
        "F1 Score": ner_metrics[2],
        "Accuracy": ner_metrics[3]
    },

    {
        "Method": "ML Model",
        "Precision": ml_metrics[0],
        "Recall": ml_metrics[1],
        "F1 Score": ml_metrics[2],
        "Accuracy": ml_metrics[3]
    },

    {
        "Method": "Transformer",
        "Precision": transformer_metrics[0],
        "Recall": transformer_metrics[1],
        "F1 Score": transformer_metrics[2],
        "Accuracy": transformer_metrics[3]
    }
])

model_comparison[
    ["Precision", "Recall", "F1 Score", "Accuracy"]
] = model_comparison[
    ["Precision", "Recall", "F1 Score", "Accuracy"]
].round(3)

model_comparison

,Method,Precision,Recall,F1 Score,Accuracy
0,Dictionary,1.000,1.000,1.000,1.000
1,Regex,0.962,0.977,0.969,0.991
2,TF-IDF,1.000,0.245,0.393,0.895
3,NER,1.000,0.021,0.040,0.864
4,ML Model,1.000,1.000,1.000,1.000
5,Transformer,1.000,0.840,0.913,0.978


In [74]:
model_comparison.to_csv(
    "model_comparison.csv",
    index=False
)

print("model_comparison.csv saved successfully")

model_comparison.csv saved successfully


In [75]:
print(model_comparison)

        Method  Precision  Recall  F1 Score  Accuracy
0   Dictionary      1.000   1.000     1.000     1.000
1        Regex      0.962   0.977     0.969     0.991
2       TF-IDF      1.000   0.245     0.393     0.895
3          NER      1.000   0.021     0.040     0.864
4     ML Model      1.000   1.000     1.000     1.000
5  Transformer      1.000   0.840     0.913     0.978


In [76]:
confusion_matrices = {
    "Dictionary": dictionary_metrics[4],
    "Regex": regex_metrics[4],
    "TF-IDF": tfidf_metrics[4],
    "NER": ner_metrics[4],
    "ML Model": ml_metrics[4],
    "Transformer": transformer_metrics[4]
}

for method, cm in confusion_matrices.items():

    print("\n" + method)
    print(cm)


Dictionary
[[2412    0]
 [   0  388]]

Regex
[[2397   15]
 [   9  379]]

TF-IDF
[[2412    0]
 [ 293   95]]

NER
[[2412    0]
 [ 380    8]]

ML Model
[[2412    0]
 [   0  388]]

Transformer
[[2412    0]
 [  62  326]]
